# 數位控制系統第三章：取樣與重建（教學版 Notebook）

本 Notebook 是 `chp3.md` 教材的教學版，額外補充：

- 每個第一次出現的 MATLAB / Octave 函數（`stairs`, `stem`, `linspace`, `abs`, `angle`, `exp`, ...）的逐步解說
- 複數運算、逐元素運算 `./` `.^`、`0/0` 陷阱等初學者容易卡住的語法
- 公式 → 程式碼的逐項對照（每段程式都標註對應的教材式號）
- 可直接執行的九個實驗，親手驗證本章每一條公式

建議搭配 `chp3.md`（完整理論、推導與符號定義）與 `chp3.m`（精簡可執行版）一起閱讀。

> **注意**：原教材第 3 章**沒有提供 MATLAB 程式碼**。本 Notebook 的所有程式都是為了驗證章節公式而補充的，不引入教材以外的新理論或新系統。

---


## 🔧 環境設定

在 Octave Jupyter Notebook 中繪圖，需要：

1. `%plot --format svg` — Octave kernel 的 magic 語法（**必須放在 cell 第一行**），讓圖表內嵌顯示；SVG 對中文字型支援較好。
2. `pkg load control` — 載入 control 套件。
3. 設定中文字型以避免亂碼。

> **需要 Control System Toolbox**（Octave 為 `control` 套件）。本章的實驗其實大多只用到基本繪圖與複數運算，但為了與其他章節一致仍先載入。
>
> 下面的 `if exist('OCTAVE_VERSION','builtin')` 判斷讓同一段程式在 **MATLAB 與 Octave 都能直接執行**——MATLAB 會自動跳過整段。


In [ ]:
%plot --format svg

if exist('OCTAVE_VERSION', 'builtin')
    warning('off', 'Octave:gnuplot-graphics');
    warning('off', 'Octave:fltk-graphics');
    graphics_toolkit('gnuplot');
    pkg load control;
end
clear; clc;

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

%% 全章共同參數
T  = 0.1;              % 取樣週期 (s)         <- 教材符號 T
ws = 2*pi/T;           % 取樣角頻率 (rad/s)    <- ws = 2*pi/T

printf('取樣週期 T   = %g s\n', T);
printf('取樣角頻率 ws = 2*pi/T = %.4f rad/s (= %.2f Hz)\n', ws, ws/(2*pi));

## 📖 全章符號總表

| 符號 | 型別 | 意義 | 單位 |
|---|---|---|---|
| $e(t)$ | 純量函數 | 進入取樣器的連續時間訊號 | 視物理量而定 |
| $T$ | 純量 | **取樣週期** | s |
| $e(kT)$ | 純量 | 第 $k$ 個取樣瞬間的訊號值 | — |
| $e^*(t)$ | 廣義函數 | 理想取樣器輸出（脈衝串），**非物理訊號** | — |
| $E^*(s)$ | 複變函數 | **星號轉換**，$e^*(t)$ 的拉氏轉換 | — |
| $\bar{e}(t)$ | 純量函數 | ZOH 輸出（階梯狀），**是物理訊號** | — |
| $\delta(t)$ | 廣義函數 | 單位脈衝函數 | — |
| $\delta_T(t)$ | 廣義函數 | 週期脈衝串 | — |
| $\omega$ | 純量 | 角頻率 | rad/s |
| $\omega_s$ | 純量 | **取樣角頻率** $=2\pi/T$ | rad/s |
| $E(j\omega)$ | 複變函數 | 傅立葉轉換（**頻譜**） | — |
| $G_{h0}(s)$ | 複變函數 | 零階保持器（ZOH） | — |
| $G_{h1}(s)$ | 複變函數 | 一階保持器（FOH） | — |
| $G_{hk}(s)$ | 複變函數 | 分數階保持器，$0\le k\le1$ | — |

## 📖 本章會用到的 Octave 語法小抄

| 語法 | 意義 | 為什麼本章需要 |
|---|---|---|
| `a:b:c` | 等差向量，從 `a` 每次加 `b` 到 `c` | 產生取樣時刻 `0:T:1` |
| `linspace(a,b,n)` | 從 `a` 到 `b` 平均取 `n` 點 | 產生頻率軸 |
| `plot(x,y)` | 折線圖 | 畫連續訊號 |
| **`stairs(x,y)`** | **階梯圖** | **畫 ZOH 輸出——用 `plot` 會誤導成連續變化** |
| **`stem(x,y)`** | **針狀圖** | **畫脈衝串 $e^*(t)$，長度代表權重** |
| `exp(x)` | 自然指數 $e^x$ | 寫 $e^{-Ts}$ 要用 `exp(-T*s)`，**不是** `e^(-T*s)` |
| `1j` 或 `i` | 虛數單位 | 計算 $G(j\omega)$ |
| `abs(z)` | 複數的絕對值 | 求振幅響應 $| G|$ |
| `angle(z)` | 複數的輻角（**弧度**） | 求相位；要轉度數需 `*180/pi` |
| **`./`** | **逐元素除法** | `sin(x)./x` 對整個向量逐點相除 |
| **`.^`** | **逐元素次方** | `(...).^2` |
| `/` `^` | 矩陣除法 / 矩陣次方 | 本章**不要**用這兩個處理向量 |
| `pi` | 內建常數 $\pi$ | $\omega_s=2\pi/T$ |
| `real(z)`, `imag(z)` | 取實部、虛部 | 檢查複數結果 |

> **⚠️ 最容易犯的錯**：`sin(x)/x` 與 `sin(x)./x` 完全不同。前者是矩陣除法（會得到一個純量），後者才是我們要的「逐點相除」。**只要 `x` 是向量，除號前面就要加點。**

---


## 二、取樣資料控制系統 (3.2)

### 概念

雷達每 $T$ 秒才發射一次，所以誤差 $e(t)$ **只在每隔 $T$ 秒才已知**。但功率放大器必須每一刻都有輸入。若直接送窄脈衝串，訊號含大量高頻成分，會激發受控體不想要的動態——因此必須加**資料保持器**。

最簡單也最常用的是**零階保持器**（ZOH）：把輸出**夾持**在取樣瞬間的值，直到下一個取樣瞬間才更新。

### 關鍵推導：式 (3-1) → 式 (3-2)

階梯訊號可寫成一連串方窗的疊加（式 3-1）：

$$\bar{e}(t)=e(0)[u(t)-u(t-T)]+e(T)[u(t-T)-u(t-2T)]+\cdots$$

取拉氏轉換（用 $\mathcal{L}[u(t-a)]=e^{-as}/s$）並提出共同因子，得到**全章最重要的分解**（式 3-2）：

$$\bar{E}(s)=\underbrace{\left[\sum_{n=0}^{\infty}e(nT)e^{-nTs}\right]}_{\text{只和訊號有關 }\to\ E^*(s)}\underbrace{\left[\frac{1-e^{-Ts}}{s}\right]}_{\text{只和 }T\text{ 有關 }\to\ G_{h0}(s)}$$

```text
       ┌─────────────┐         ┌──────────────────┐
e(t)   │  理想取樣器  │  E*(s)  │  (1 - e^{-Ts})/s │   ē(t)
─────► │   （開關）   ├────────►│   資料保持器      ├──────►
E(s)   └─────────────┘         └──────────────────┘
```

**一個實體裝置，數學上恰好拆成兩個獨立方塊。** 這個拆法讓「取樣」與「保持」可以分開分析——這就是第 3.3 節與第 3.7 節分頭進行的原因。

### 這段程式用到的語法

| 程式 | 意義 |
|---|---|
| `t_fine = 0:T/200:1;` | 每 `T/200` 秒取一點，密到看起來像連續曲線 |
| `t_k = 0:T:1;` | **真正的取樣時刻** $t=0,T,2T,\dots$ |
| `stairs(t_k, e_k)` | 畫成階梯——**這才是 ZOH 的正確畫法** |
| `'ko'` | 黑色圓形標記（`k`=black，`o`=circle） |
| `'MarkerFaceColor','k'` | 把圓形填滿 |


In [ ]:
%% 實驗 1：取樣器 + 零階保持器的時域波形（式 3-28，對應 Fig. 3-3）
t_fine = 0:T/200:1;                 % 密取樣，用來畫「連續」原訊號
e_fine = sin(2*pi*1.5*t_fine);      % 原訊號 e(t) = sin(2*pi*1.5*t)

t_k = 0:T:1;                        % 取樣瞬間 t = kT
e_k = sin(2*pi*1.5*t_k);            % 取樣值 e(kT)

figure('Position', [50 50 800 380]);
plot(t_fine, e_fine, 'b-', 'LineWidth', 1.5); hold on;
stairs(t_k, e_k, 'r-', 'LineWidth', 2);
plot(t_k, e_k, 'ko', 'MarkerFaceColor', 'k', 'MarkerSize', 5);
grid on;
title('取樣器 + 零階保持器（式 3-28）');
xlabel('時間 t (秒)'); ylabel('振幅');
legend('原訊號 e(t)', 'ZOH 輸出（階梯狀）', '取樣值 e(kT)', 'Location', 'southwest');

printf('共 %d 個取樣點，取樣間隔 T = %g 秒\n', numel(t_k), T);

### 結果解讀

紅色階梯就是**實際送進受控體的訊號**。注意兩件事：

1. **階梯永遠落後於原訊號**——因為 ZOH 用的是「上一個取樣瞬間」的值，在整個區間內都不更新。這個落後平均是半個取樣週期，也就是 $T/2$（實驗 9 會嚴格驗證）。
2. **階梯的轉角是不連續的**——這些不連續正是高頻成分的來源，也是 ZOH 頻率響應在高頻不會歸零的原因。

**若把 `stairs` 換成 `plot` 會怎樣？** 你會看到一條把取樣點連起來的折線——那**不是**系統實際輸出的訊號，會讓你誤以為訊號在取樣點之間平滑變化。**數位控制的輸出一律用 `stairs` 畫。**

---


## 三、理想取樣器 (3.3)

### 概念與公式

對式 (3-3) 取反拉氏轉換（式 3-4）：

$$e^*(t)=e(0)\delta(t)+e(T)\delta(t-T)+e(2T)\delta(t-2T)+\cdots$$

$e^*(t)$ 是一串**脈衝**，每根脈衝的**權重（面積）等於訊號在該瞬間的值**。

> **⚠️ 三個必須記住的觀念**
> 1. 脈衝的振幅是**無限大**，所以圖上箭頭長度代表的是**權重**，不是振幅。
> 2. $e^*(t)$ **不是物理訊號**，實體系統裡量不到它。
> 3. 正因為輸出是非物理的脈衝，這個取樣器才叫**理想取樣器**（ideal sampler），也叫**脈衝調變器**。

### 取樣就是調變（式 3-5、3-6）

$$\delta_T(t)=\sum_{n=0}^{\infty}\delta(t-nT),\qquad e^*(t)=e(t)\,\delta_T(t)$$

| 調變術語 | 對應 |
|---|---|
| **載波** | $\delta_T(t)$（週期脈衝串） |
| **調變訊號** | $e(t)$ |
| **已調變訊號** | $e^*(t)$ |

**為什麼這個觀點重要？** 調變在頻域的效果是「把頻譜搬到載波頻率複製一份」。載波是間隔 $T$ 的脈衝串，其頻譜也是間隔 $\omega_s$ 的脈衝串——所以**取樣的頻域效果就是把原頻譜以 $\omega_s$ 為間隔複製無限多份**。這是第 3.6 節混疊現象的根源。

### 為什麼用 `stem` 而不是 `plot`

`stem(x, y)` 畫出「從 0 拉一根針到 $(x,y)$，頂端加一個圓點」的圖。這正好對應脈衝串的慣用畫法：**針的長度代表權重**。`'filled'` 選項把頂端的圓點填滿。


In [ ]:
%% 實驗 2：理想取樣器的輸出 e*(t)（式 3-4，對應 Fig. 3-6）
figure('Position', [50 50 800 380]);
stem(t_k, e_k, 'filled', 'LineWidth', 1.5); hold on;
plot(t_fine, e_fine, 'b:', 'LineWidth', 1);
grid on;
title('理想取樣器輸出 e^*(t)：脈衝串（式 3-4）');
xlabel('時間 t (秒)'); ylabel('脈衝權重 e(nT)');
legend('e^*(t) 的脈衝權重', '原訊號 e(t)', 'Location', 'southwest');

printf('每根針的長度 = 該瞬間的訊號值 e(nT)，不是脈衝的振幅（振幅為無限大）\n');
printf('前 5 個權重： ');
printf('%.4f  ', e_k(1:5)); printf('\n');

### 結果解讀

把實驗 1 和實驗 2 放在一起看，就是式 (3-2) 那個分解的兩半：

```text
實驗 2 的脈衝串  ──►  經過 (1-e^{-Ts})/s  ──►  實驗 1 的階梯
   e*(t)                  資料保持器                 ē(t)
（數學中間量）                                    （物理訊號）
```

**中間那個脈衝串在真實電路裡不存在**，它純粹是把式 (3-2) 因式分解後跑出來的數學物件。但「脈衝串 + 保持器」串起來，確實準確描述了實體取樣保持裝置的輸入輸出行為。

---


## 範例 3.1：單位步階的星號轉換

### 公式

$e(t)=u(t)$，所以 $e(nT)=1$ 對所有 $n\ge0$。代入定義（式 3-7）：

$$E^*(s)=\sum_{n=0}^{\infty}1\cdot e^{-nTs}=1+e^{-Ts}+e^{-2Ts}+\cdots$$

這是**幾何級數**。用公式 $\dfrac{1}{1-x}=1+x+x^2+\cdots$（收斂條件 $| x|<1$），令 $x=e^{-Ts}$：

$$\boxed{E^*(s)=\frac{1}{1-e^{-Ts}},\qquad \left| e^{-Ts}\right|<1}$$

### 數學 ↔ MATLAB 對照

| 數學 | 程式 | 說明 |
|---|---|---|
| $e^{-nTs}$ | `exp(-n*T*s)` | **`exp()` 不是 `e^`** |
| $\sum_{n=0}^{N}$ | `sum(...)` 配合 `n = 0:N` | `n` 是向量，整個式子一次算完 |
| $s=2+3j$ | `s = 2 + 1j*3;` | `1j` 是虛數單位 |
| $| e^{-Ts}|$ | `abs(exp(-T*s))` | 收斂條件要 < 1，所以取 $\mathrm{Re}(s)>0$ |

> **為什麼要挑 $\mathrm{Re}(s)>0$ 的測試點？** 因為 $| e^{-Ts}|=e^{-T\,\mathrm{Re}(s)}$，只有 $\mathrm{Re}(s)>0$ 時它才小於 1，級數才收斂。這正是封閉形式後面那個條件的意思。


In [ ]:
%% 實驗 3：星號轉換 —— 無窮級數 vs 封閉形式（例 3.1）
s_test = 2 + 1j*3;                          % 任取一個 Re(s)>0 的測試點
N = 2000;                                   % 級數截斷項數
n = 0:N;                                    % n = 0,1,2,...,N（列向量）

E_series = sum(1 .* exp(-n*T*s_test));      % 式 (3-3)：sum e(nT)*exp(-n*T*s)，e(nT)=1
E_closed = 1 / (1 - exp(-T*s_test));        % 封閉形式

printf('測試點 s = %g %+gj\n', real(s_test), imag(s_test));
printf('  |exp(-T*s)| = %.6f   (需 < 1 級數才收斂)\n\n', abs(exp(-T*s_test)));
printf('  級數前 %d 項 = %.10f %+.10fj\n', N+1, real(E_series), imag(E_series));
printf('  封閉形式     = %.10f %+.10fj\n', real(E_closed), imag(E_closed));
printf('  誤差         = %.2e\n', abs(E_series - E_closed));

### 結果解讀與對照第 4 章

誤差在 $10^{-15}$ 等級，**封閉形式與無窮級數完全一致**。

這個結果直接連到第 4 章的核心關係（式 4-3）。把 $z=e^{Ts}$ 代入：

$$E^*(s)=\frac{1}{1-e^{-Ts}}=\frac{1}{1-z^{-1}}=\frac{z}{z-1}$$

**這正是單位步階的 z 轉換。** 第 4 章會證明：

$$E(z)=E^*(s)\Big|_{e^{sT}=z}$$

也就是說，**z 轉換就是星號轉換換個變數寫法**——這也是為什麼教科書通常不另外列星號轉換表，直接用 z 轉換表就好。

> **試試看**：把 `s_test` 改成 `-1 + 1j*3`（$\mathrm{Re}(s)<0$），觀察級數會發散、兩者不再相等。

---


## 四、$E^*(s)$ 的求法 (3.4) 與傅立葉轉換 (3.5)

這兩節主要是**理論準備**，沒有直接對應的數值實驗，但公式在後面會用到，先列出來。

### 3.4 兩個額外的表示式

定義式 (3-3) 是無窮級數，分析上不好用。教材另給兩式：

**留數法（式 3-10）**——用來產生 z 轉換表：

$$E^*(s)=\sum_{\text{at poles of }E(\lambda)}\left[\text{residues of }E(\lambda)\frac{1}{1-e^{-T(s-\lambda)}}\right]$$

**頻率位移和（式 3-11）**——**第 3.6 節的一切都從這一式推出**：

$$E^*(s)=\frac{1}{T}\sum_{n=-\infty}^{\infty}E(s+jn\omega_s)+\frac{e(0)}{2}$$

> 這一式直接說出「取樣造成頻譜複製」：把原頻譜沿虛軸每隔 $\omega_s$ 複製一份再全部相加，並乘上 $1/T$。$e(0)/2$ 那一項是為了處理 $t=0$ 不連續的情形；若 $e(t)$ 在所有取樣瞬間都連續，這項為零。

**範例 3.3**：$E(s)=\dfrac{1}{(s+1)(s+2)}$ 有兩個單極點，分別取留數相加得

$$E^*(s)=\frac{1}{1-e^{-T(s+1)}}-\frac{1}{1-e^{-T(s+2)}}$$

### 3.5 傅立葉轉換：提供頻域的語言

$$\mathcal{F}[e(t)]=E(j\omega)=\int_{-\infty}^{\infty}e(t)e^{-j\omega t}dt$$

若 $e(t)$ 在 $t<0$ 為零，則

$$\mathcal{F}[e(t)u(t)]=\mathcal{L}[e(t)u(t)]\Big|_{s=j\omega}$$

**這解釋了為什麼傅立葉轉換要寫成 $E(j\omega)$ 而不是 $E(\omega)$**——它就是拉氏轉換把 $s$ 換成 $j\omega$。

$E(j\omega)$ 的圖形稱為**頻譜**：$| E(j\omega)|$ 對 $\omega$ 是**振幅頻譜**，$\angle E(j\omega)$ 對 $\omega$ 是**相位頻譜**。而 $G(j\omega)$ 稱為系統的**頻率響應**，它決定「輸入頻譜怎麼被改造成輸出頻譜」。

**這正是實驗 6～8 要畫的東西**：$G_{h0}(j\omega)$、$G_{h1}(j\omega)$、$G_{hk}(j\omega)$ 就是三種保持器的頻率響應。

---


## 五、$E^*(s)$ 的性質 (3.6)

### 性質 1：對 $s$ 週期，週期為 $j\omega_s$

$$E^*(s+jm\omega_s)=E^*(s),\qquad m\text{ 為整數}$$

**證明**：由定義式 (3-3)，關鍵在 $\omega_sT=\dfrac{2\pi}{T}T=2\pi$，因此

$$e^{-jnm\omega_sT}=e^{-jnm2\pi}=1$$

該因子恆為 1，級數不變。

### 性質 2：極點以 $j\omega_s$ 為間隔複製

$$E(s)\text{ 在 }s=s_1\text{ 有極點}\ \Longrightarrow\ E^*(s)\text{ 在 }s=s_1+jm\omega_s\text{ 都有極點}$$

由式 (3-11) 的每一項各貢獻一個極點即得。

> **⚠️ 零點沒有對應結論**：$E(s)$ 的零點位置**無法**唯一決定 $E^*(s)$ 的零點位置（但零點仍以 $j\omega_s$ 為週期重複，這是性質 1 的結果）。

### 主帶（primary strip）

$$-\frac{\omega_s}{2}\le\omega\le\frac{\omega_s}{2}$$

由性質 1，**只要知道主帶內的極零點，整個 $s$ 平面的極零點就全確定了**。

### 混疊的起源：兩個訊號、同一組取樣值（Fig. 3-9）

取 $\omega_1=\omega_s/4$，比較 $e_1(t)=\cos\omega_1t$ 與 $e_2(t)=\cos3\omega_1t$：

| 訊號 | 極點 |
|---|---|
| $\cos\omega_1t$ | $s=\pm j\omega_1$ |
| $\cos3\omega_1t$ | $s=\pm j3\omega_1$ |

而 $j3\omega_1=j(-\omega_1+\omega_s)$——**兩組極點恰好相差 $\omega_s$**。由性質 2，它們產生完全相同的 $E^*(s)$ 極點。下一格用數值驗證。


In [ ]:
%% 實驗 4：兩個訊號、同一組取樣值（Fig. 3-9）
w1 = ws/4;                       % 取 w1 = ws/4，則 3*w1 = ws - w1

t_c  = 0:T/300:6*T;              % 密取樣畫連續曲線
y1   = cos(w1*t_c);              % e1(t) = cos(w1*t)
y2   = cos(3*w1*t_c);            % e2(t) = cos(3*w1*t)

tk_c = 0:T:6*T;                  % 取樣瞬間
y1k  = cos(w1*tk_c);
y2k  = cos(3*w1*tk_c);

figure('Position', [50 50 800 380]);
plot(t_c, y1, 'b-', 'LineWidth', 1.5); hold on;
plot(t_c, y2, 'r--', 'LineWidth', 1.5);
plot(tk_c, y1k, 'ko', 'MarkerFaceColor', 'k', 'MarkerSize', 8);
grid on;
title('cos(\omega_1t) 與 cos(3\omega_1t) 取樣值相同（Fig. 3-9）');
xlabel('時間 t (秒)'); ylabel('振幅');
legend('e_1(t)=cos(\omega_1 t)', 'e_2(t)=cos(3\omega_1 t)', '共同的取樣值', ...
       'Location', 'southeast');

printf('w1 = ws/4 = %.4f rad/s，3*w1 = %.4f rad/s\n', w1, 3*w1);
printf('ws - w1 = %.4f rad/s   <- 與 3*w1 相同\n\n', ws - w1);
printf('  k    t=kT      cos(w1*t)    cos(3*w1*t)    差\n');
for k = 1:5
    printf('  %-4d %-9.2f %-12.6f %-14.6f %.2e\n', ...
           k-1, tk_c(k), y1k(k), y2k(k), abs(y1k(k)-y2k(k)));
end
printf('\n兩組取樣值最大差異 = %.2e  => 取樣後完全無法分辨\n', max(abs(y1k - y2k)));

### 結果解讀

兩條完全不同的曲線（藍實線振盪慢、紅虛線振盪快 3 倍），**在每一個黑點上的值都一模一樣**，差異只有浮點誤差等級。

> **💡 白話文**：取樣之後，**高頻訊號會偽裝成低頻訊號**。$3\omega_1$ 的餘弦波每隔 $T$ 秒看一眼，看起來就跟 $\omega_1$ 的餘弦波完全相同——你**無法從取樣值分辨它們**。

這就是**混疊**（aliasing）。從極點看：$3\omega_1$ 與 $-\omega_1+\omega_s$ 是同一個數，由性質 2 它們對應相同的 $E^*(s)$ 極點。

---


## Shannon 取樣定理

### 頻譜複製（式 3-22）

把式 (3-21) 在 $s=j\omega$ 求值：

$$E^*(j\omega)=\frac{1}{T}\Big[E(j\omega)+E(j\omega+j\omega_s)+E(j\omega-j\omega_s)+\cdots\Big]+\frac{e(0)}{2}$$

> **理想取樣的效果 = 把原頻譜複製到 $\pm\omega_s,\pm2\omega_s,\dots$ 並全部疊加。**

### 定理

> **Shannon 取樣定理**：一個訊號 $e(t)$，若其傅立葉轉換不含高於 $f_0$ Hz 的成分，則 $e(t)$ 可由**間隔 $\dfrac{1}{2f_0}$ 秒**的任一組取樣點唯一決定。

**幾何直覺**：若原訊號最高頻率 $<\omega_s/2$，各複製版本**不重疊**，用理想低通濾波器就能完整還原；若 $>\omega_s/2$，各版本**重疊**，此時**無論用什麼濾波器都救不回來**。

**實務準則**：取樣頻率應大於「訊號中具有顯著振幅的最高頻率」的**兩倍**。

### 混疊後看起來是幾 Hz？

若真實頻率 $f_0$ 被以 $f_s$ 取樣且 $f_s<2f_0$，摺返後的視在頻率為 $| f_0-kf_s|$ 中落在 $[0,f_s/2]$ 的那一個。下面的例子 $f_0=3$、$f_s=4$，視在頻率 $=|3-4|=1$ Hz。


In [ ]:
%% 實驗 5：Shannon 取樣定理 —— 足夠 vs 不足
f_sig  = 3;                        % 訊號頻率 3 Hz
t_true = 0:0.0005:1;
x_true = sin(2*pi*f_sig*t_true);

fs_ok  = 20;   Ts_ok  = 1/fs_ok;   % 20 Hz > 2*3 Hz，足夠
fs_bad = 4;    Ts_bad = 1/fs_bad;  % 4 Hz  < 2*3 Hz，不足 -> 混疊

tk_ok  = 0:Ts_ok:1;    xk_ok  = sin(2*pi*f_sig*tk_ok);
tk_bad = 0:Ts_bad:1;   xk_bad = sin(2*pi*f_sig*tk_bad);
f_alias = abs(f_sig - fs_bad);     % 摺返後的視在頻率

figure('Position', [50 50 850 620]);
subplot(2,1,1);
plot(t_true, x_true, 'b-', 'LineWidth', 1.2); hold on;
stem(tk_ok, xk_ok, 'r', 'filled'); grid on;
title(sprintf('取樣率足夠：f_s = %g Hz > 2f_0 = %g Hz', fs_ok, 2*f_sig));
xlabel('時間 (秒)'); ylabel('振幅');

subplot(2,1,2);
plot(t_true, x_true, 'b-', 'LineWidth', 1.2); hold on;
stem(tk_bad, xk_bad, 'r', 'filled');
plot(t_true, sin(2*pi*f_alias*t_true), 'g--', 'LineWidth', 1.5); grid on;
title(sprintf('取樣率不足：f_s = %g Hz < 2f_0 = %g Hz，看起來變成 %g Hz', ...
      fs_bad, 2*f_sig, f_alias));
xlabel('時間 (秒)'); ylabel('振幅');
legend('真實 3 Hz 訊號', '取樣值', sprintf('混疊後的 %g Hz', f_alias), ...
       'Location', 'southwest');

printf('真實頻率 f0 = %g Hz\n', f_sig);
printf('  fs = %g Hz：Nyquist 頻率 = %g Hz > f0，安全\n', fs_ok, fs_ok/2);
printf('  fs = %g Hz：Nyquist 頻率 = %g Hz < f0，混疊成 |f0-fs| = %g Hz\n', ...
       fs_bad, fs_bad/2, f_alias);

### 結果解讀

**上圖**：取樣點密集分布在正弦波上，把它們連起來能明顯看出 3 Hz 的形狀——資訊沒有遺失。

**下圖**：取樣點**恰好落在一條 1 Hz 正弦波上**（綠色虛線）。從這些取樣值出發，你會斷定訊號是 1 Hz——**3 Hz 的資訊已經永久遺失，任何後處理都救不回來**。

> **這對控制系統的意義**：感測器雜訊、機械共振等高頻成分若沒有先濾掉就取樣，會被混疊成低頻，**被控制器誤判成真正的誤差而做出錯誤修正**。這就是為什麼實務上要在取樣器**前面**加**類比防混疊濾波器**（antialiasing filter）。
>
> **但防混疊濾波器不能無限降低截止頻率**——低通濾波器會引入相位落後，截止頻率太低會讓閉迴路系統不穩定。這是一個必須權衡的設計取捨。

---


## 六、資料重建 (3.7)：零階保持器

### 泰勒外插的統一觀點（式 3-23 ~ 3-27）

把 $e(t)$ 在 $t=nT$ 附近做泰勒展開：

$$e(t)=e(nT)+e'(nT)(t-nT)+\frac{e''(nT)}{2!}(t-nT)^2+\cdots$$

**問題**：訊號只以取樣形式進入保持器，**導數未知**。**解法**：用**後向差分**近似：

$$e'(nT)=\frac{e(nT)-e[(n-1)T]}{T}$$

**三種保持器的差別，只在於泰勒級數取幾項**：

| 保持器 | 取到第幾項 | 外插形狀 | 需要記憶體 |
|---|---|---|---|
| ZOH | 只取常數項 | 水平線（階梯） | 否 |
| FOH | 取到一次項 | 斜直線 | 是 |
| 分數階 | 一次項乘 $k$ | 斜率打折 | 是 |

### ZOH 轉移函數（式 3-29）

用「輸入是單位脈衝」的技巧推導：脈衝進去，輸出是寬 $T$ 高 1 的方波

$$e_o(t)=u(t)-u(t-T)\ \Longrightarrow\ E_o(s)=\frac{1}{s}-\frac{e^{-Ts}}{s}$$

因為 $E_i(s)=1$（脈衝的拉氏轉換）：

$$\boxed{G_{h0}(s)=\frac{1-e^{-Ts}}{s}}$$

### 頻率響應的推導（式 3-30 ~ 3-33）

令 $s=j\omega$，用「提出半角湊尤拉公式」的技巧：

$$G_{h0}(j\omega)=\frac{1-e^{-j\omega T}}{j\omega}\cdot\frac{e^{j\omega T/2}}{e^{j\omega T/2}}
=\frac{2e^{-j\omega T/2}}{\omega}\left[\frac{e^{j\omega T/2}-e^{-j\omega T/2}}{2j}\right]
=T\frac{\sin(\omega T/2)}{\omega T/2}e^{-j\omega T/2}$$

由於 $\dfrac{\omega T}{2}=\dfrac{\pi\omega}{\omega_s}$：

$$\boxed{G_{h0}(j\omega)=T\frac{\sin(\pi\omega/\omega_s)}{\pi\omega/\omega_s}e^{-j\pi\omega/\omega_s}}$$

$$\left|G_{h0}(j\omega)\right|=T\left|\frac{\sin(\pi\omega/\omega_s)}{\pi\omega/\omega_s}\right|,\qquad
\angle G_{h0}(j\omega)=-\frac{\pi\omega}{\omega_s}+\phi$$

其中 $\phi=0$（當 $\sin>0$）或 $\pi$（當 $\sin<0$）。

### ⚠️ 程式實作的陷阱：$0/0$

$\omega=0$ 時 $\dfrac{\sin(\pi\omega/\omega_s)}{\pi\omega/\omega_s}$ 是 $\dfrac00$，數學上極限為 1，但**程式會算出 `NaN`（Not a Number）**，圖上會出現斷點。

**兩種解法**：

1. **頻率軸從很小的正數開始**（本 Notebook 採用）：`linspace(1e-6, 3*ws, 3000)`
2. 事後補值：`sc(w==0) = 1;`

### 數學 ↔ MATLAB 對照

| 數學 | 程式 |
|---|---|
| $x=\dfrac{\pi\omega}{\omega_s}$ | `x = pi*w/ws;` |
| $\dfrac{\sin x}{x}$ | `sin(x)./x`（**注意 `./`**） |
| $T|\mathrm{sinc}|$ | `T*abs(sc)` |
| $Te^{-jx}\mathrm{sinc}$ | `T*sc.*exp(-1j*x)`（**`.*` 逐元素乘**） |
| 相位（度） | `angle(Gh0)*180/pi` |


In [ ]:
%% 實驗 6：零階保持器的頻率響應（式 3-32、3-33，對應 Fig. 3-13）
w  = linspace(1e-6, 3*ws, 3000);   % 從 1e-6 開始，避開 omega=0 的 0/0
x  = pi*w/ws;                       % 式中反覆出現的 pi*w/ws
sc = sin(x)./x;                     % sinc；注意是 ./ 逐元素除法

mag_h0 = T*abs(sc);                       % 式 (3-32)
Gh0    = T*sc.*exp(-1j*x);                % 式 (3-31) 完整複數
pha_h0 = angle(Gh0)*180/pi;               % 式 (3-33)，弧度轉度

figure('Position', [50 50 850 620]);
subplot(2,1,1);
plot(w/ws, mag_h0, 'b-', 'LineWidth', 1.5); grid on;
title('ZOH 振幅響應 |G_{h0}(j\omega)|（式 3-32）');
xlabel('\omega / \omega_s'); ylabel('振幅');
subplot(2,1,2);
plot(w/ws, pha_h0, 'b-', 'LineWidth', 1.5); grid on;
title('ZOH 相位響應（式 3-33）');
xlabel('\omega / \omega_s'); ylabel('相位 (度)');

printf('omega -> 0    : |Gh0| = %.6f   (理論值 T = %g)\n', mag_h0(1), T);
[~, iz] = min(abs(w - ws));
printf('omega = ws    : |Gh0| = %.2e   (理論上為 0，sinc 的零點)\n', mag_h0(iz));
[~, ih] = min(abs(w - ws/2));
printf('omega = ws/2  : |Gh0| = %.6f   相位 = %.2f 度\n', mag_h0(ih), pha_h0(ih));

### 結果解讀

**振幅**：一條 sinc 曲線。

- $\omega\to0$ 時值為 $T$（最大值）
- 在 $\omega=k\omega_s$（$k$ 為非零整數）處為**零**——所以當訊號頻率遠低於 $\omega_s/2$ 時，取樣產生的高頻鏡像剛好落在這些零點附近，**ZOH 對訊號的影響很小**
- 在 Nyquist 頻率 $\omega_s/2$ 處衰減到 $2T/\pi\approx0.637T$

**相位**：主要是一條**線性下降**的直線 $-\pi\omega/\omega_s$，中間夾雜 $180^\circ$ 的跳變（那是振幅取絕對值時把 $\sin$ 的負號搬到相位造成的）。

> **💡 這條線性相位是 ZOH 對控制系統最大的傷害**：它等效於**半個取樣週期的純延遲**（實驗 9 會嚴格驗證）。$T$ 越大，相位落後越嚴重，閉迴路的相位裕度就被吃掉越多，系統越容易不穩定。**這是「取樣週期不能取太大」的另一個理由**——不只是 Shannon 定理的資訊考量，還有穩定度考量。

---


## 七、一階保持器 (FOH)

### 定義與轉移函數（式 3-34 ~ 3-36）

取泰勒展開前兩項，用**前一區間的斜率**做線性外插：

$$e_n(t)=e(nT)+e'(nT)(t-nT),\qquad e'(nT)=\frac{e(nT)-e[(n-1)T]}{T}$$

> **需要記憶體**：在 $t=nT$ 時必須取用 $e[(n-1)T]$。這是 FOH 比 ZOH 貴的原因。

同樣假設輸入為單位脈衝，推導後得：

$$\boxed{G_{h1}(s)=\frac{1+Ts}{T}\left[\frac{1-e^{-Ts}}{s}\right]^{2}}$$

**結構解讀**：括號裡就是 ZOH，所以 **FOH = ZOH 的平方，再乘上 $\dfrac{1+Ts}{T}$**。

### 頻率響應（式 3-37、3-38）

$$\left|G_{h1}(j\omega)\right|=T\sqrt{1+\frac{4\pi^2\omega^2}{\omega_s^2}}\left[\frac{\sin(\pi\omega/\omega_s)}{\pi\omega/\omega_s}\right]^{2}$$

$$\angle G_{h1}(j\omega)=\tan^{-1}\!\left(\frac{2\pi\omega}{\omega_s}\right)-\frac{2\pi\omega}{\omega_s}$$

### 語法提醒

| 數學 | 程式 |
|---|---|
| $\sqrt{\cdot}$ | `sqrt(...)` |
| $\omega^2$（向量逐元素） | `w.^2`（**`.^` 不是 `^`**） |
| $[\mathrm{sinc}]^2$ | `sc.^2` |
| $\tan^{-1}$ | `atan(...)`（**不是 `tan(...)^-1`**） |


In [ ]:
%% 實驗 7：FOH 頻率響應，與 ZOH 比較（式 3-37、3-38，對應 Fig. 3-17）
mag_h1 = T*sqrt(1 + 4*pi^2*w.^2./ws^2).*(sc).^2;    % 式 (3-37)
pha_h1 = (atan(2*pi*w/ws) - 2*pi*w/ws)*180/pi;       % 式 (3-38)

figure('Position', [50 50 850 620]);
subplot(2,1,1);
plot(w/ws, mag_h0, 'b-', 'LineWidth', 1.5); hold on;
plot(w/ws, mag_h1, 'r--', 'LineWidth', 1.5); grid on;
title('振幅響應：ZOH vs FOH');
xlabel('\omega / \omega_s'); ylabel('振幅');
legend('ZOH |G_{h0}|（式 3-32）', 'FOH |G_{h1}|（式 3-37）', 'Location', 'northeast');

subplot(2,1,2);
plot(w/ws, pha_h0, 'b-', 'LineWidth', 1.5); hold on;
plot(w/ws, pha_h1, 'r--', 'LineWidth', 1.5); grid on;
title('相位響應：ZOH vs FOH');
xlabel('\omega / \omega_s'); ylabel('相位 (度)');
legend('ZOH（式 3-33）', 'FOH（式 3-38）', 'Location', 'southwest');

printf('理想低通濾波器：通帶 |G| = T = %g 且「相位 = 0」；阻帶 |G| = 0\n\n', T);
printf('  %-9s %-11s %-11s %-14s %s\n', 'w/ws', '|Gh0|', '|Gh1|', 'ang Gh0(deg)', 'ang Gh1(deg)');
for r = [0.05 0.10 0.30 0.50 0.60]
    [~, ii] = min(abs(w/ws - r));
    printf('  %-9.2f %-11.5f %-11.5f %-14.2f %.2f\n', ...
           r, mag_h0(ii), mag_h1(ii), pha_h0(ii), pha_h1(ii));
end

### 結果解讀：教材結論的細節

教材說「**零頻率附近 FOH 較接近理想低通濾波器，較大 $\omega$ 則 ZOH 較好**」，但沒有點明是振幅還是相位。從上表可以看清楚——**優勢在相位**：

| $\omega/\omega_s$ | $| G_{h0}|$ | $| G_{h1}|$ | $\angle G_{h0}$ | $\angle G_{h1}$ |
|---|---|---|---|---|
| 0.05 | 0.0996 | 0.1040 | $-9.00^\circ$ | $\mathbf{-0.56^\circ}$ |
| 0.10 | 0.0984 | 0.1143 | $-18.0^\circ$ | $\mathbf{-3.86^\circ}$ |
| 0.50 | 0.0637 | 0.1336 | $\mathbf{-90.0^\circ}$ | $-107.7^\circ$ |
| 0.60 | $\mathbf{0.0504}$ | 0.0992 | $\mathbf{-108^\circ}$ | $-140.9^\circ$ |

理想低通濾波器在通帶內是「振幅 $=T$ **且相位 $=0$**」。

- **低頻**：FOH 相位落後只有 ZOH 的十幾分之一（$-0.56^\circ$ vs $-9^\circ$）——**這才是「較接近理想」的真正含意**。至於振幅，低頻時反而是 ZOH 比較平坦。
- **高頻**：ZOH 全面較優。阻帶（$\omega>\omega_s/2$，理想值 0）ZOH 衰減到 0.050 而 FOH 只到 0.099；相位也是 ZOH 較佳。
- **⚠️ FOH 在中頻會放大訊號**：$\omega/\omega_s=0.3$ 時 $| G_{h1}|=0.157$，比 $T=0.1$ 高出 **57%**。

因此教材的建議是：

- 若訊號頻率 $\omega_1\ll\omega_s/2$ → FOH 重建較佳
- 若 $\omega_1$ 與 $\omega_s/2$ 同數量級 → **ZOH 反而較好**

> **但實務上 ZOH 遠比 FOH 常用，主要理由是成本**（FOH 需要記憶體與額外運算）。這也是為什麼 `c2d()` 的預設方法是 `'zoh'`。

---


## 八、分數階保持器 (Fractional-Order Hold)

### 想法與公式（式 3-39）

FOH 用「前一區間的完整斜率」外插，容易衝過頭。**只採用斜率的一部分**（乘上 $k$，$0\le k\le1$）可以降低誤差：

$$\boxed{G_{hk}(s)=(1-ke^{-Ts})\frac{1-e^{-Ts}}{s}+\frac{k}{Ts^2}\left(1-e^{-Ts}\right)^2}$$

| $k$ | 退化成 |
|---|---|
| $k=0$ | **零階保持器** |
| $k=1$ | **一階保持器** |

### 語法：直接用複數算，不必手動化簡

這次不像前兩個實驗有現成的振幅公式，我們**直接把 $s=j\omega$ 代進轉移函數**用複數運算：

```matlab
sj  = 1j*w;                                        % s = j*omega（向量）
E1  = 1 - exp(-T*sj);                              % (1 - e^{-Ts})
Ghk = (1 - k*exp(-T*sj)).*E1./sj + (k/T)*(E1.^2)./(sj.^2);
```

**每個運算子都要加點**（`.*`、`./`、`.^`），因為 `w` 是向量、要逐點計算。最後用 `abs(Ghk)` 取振幅。

> **這個技巧很通用**：只要有轉移函數 $G(s)$ 的表達式，把 `s = 1j*w` 代進去、逐元素運算、再取 `abs` 和 `angle`，就能畫出任何系統的頻率響應——不需要先手動推導振幅與相位公式。


In [ ]:
%% 實驗 8：分數階保持器（式 3-39，對應 Fig. 3-19）
figure('Position', [50 50 850 450]);
k_list = [0 0.2 0.6 1];
colors = {'b', 'g', 'm', 'r'};
hold on;
for ii = 1:numel(k_list)
    k   = k_list(ii);
    sj  = 1j*w;                                  % s = j*omega
    E1  = 1 - exp(-T*sj);                        % (1 - e^{-Ts})
    Ghk = (1 - k*exp(-T*sj)).*E1./sj + (k/T)*(E1.^2)./(sj.^2);   % 式 (3-39)
    plot(w/ws, abs(Ghk), colors{ii}, 'LineWidth', 1.5);
end
grid on;
title('分數階保持器振幅響應 |G_{hk}(j\omega)|（式 3-39，對應 Fig. 3-19）');
xlabel('\omega / \omega_s'); ylabel('振幅');
legend('k = 0（即 ZOH）', 'k = 0.2', 'k = 0.6', 'k = 1（即 FOH）', 'Location', 'northeast');

% 驗證兩個端點確實退化成 ZOH 與 FOH
sj = 1j*w; E1 = 1 - exp(-T*sj);
Ghk0 = (1 - 0*exp(-T*sj)).*E1./sj + (0/T)*(E1.^2)./(sj.^2);
Ghk1 = (1 - 1*exp(-T*sj)).*E1./sj + (1/T)*(E1.^2)./(sj.^2);
printf('端點驗證：\n');
printf('  k=0 的 |Ghk| 與 ZOH 式(3-32) 最大差異 = %.2e\n', max(abs(abs(Ghk0) - mag_h0)));
printf('  k=1 的 |Ghk| 與 FOH 式(3-37) 最大差異 = %.2e\n', max(abs(abs(Ghk1) - mag_h1)));

### 結果解讀

四條曲線從 $k=0$（ZOH）平滑過渡到 $k=1$（FOH）：

- $k$ 越大，**低頻越平坦**（越接近理想低通的通帶）
- 但 $k$ 越大，**中頻的隆起也越嚴重**（訊號被放大）

端點驗證的差異在 $10^{-16}$ 等級——**式 (3-39) 在 $k=0$ 與 $k=1$ 確實分別退化成式 (3-32) 與式 (3-37)**，證明三個公式是自洽的。

> **實務意義**：雖然除了特定情況外很難決定最佳的 $k$，但分數階保持器提供了一個**連續可調的旋鈕**，可以把保持器的頻率響應「配」到被取樣訊號的頻譜上，產生誤差最小的外插。

---


## 九、ZOH 等效於 $T/2$ 的純延遲

### 為什麼要單獨驗證這件事

這是 ZOH 對控制系統設計**最重要的實務影響**。從式 (3-31)：

$$G_{h0}(j\omega)=T\,\mathrm{sinc}\!\left(\frac{\pi\omega}{\omega_s}\right)e^{-j\pi\omega/\omega_s}$$

那個相位因子 $e^{-j\pi\omega/\omega_s}$，因為 $\dfrac{\pi}{\omega_s}=\dfrac{\pi}{2\pi/T}=\dfrac{T}{2}$，可以寫成：

$$e^{-j\omega T/2}$$

**這正是「純延遲 $T/2$ 秒」的頻率響應**（時域延遲 $\tau$ 對應頻域乘上 $e^{-j\omega\tau}$）。

$$\boxed{\text{ZOH 在相位上等效於一個 }T/2\text{ 秒的純延遲}}$$


In [ ]:
%% 實驗 9：ZOH 的 T/2 等效延遲
pha_zoh_linear = -(pi*w/ws)*180/pi;      % ZOH 的線性相位項（式 3-33 的第一項）
pha_pure_delay = -(w*T/2)*180/pi;        % 純延遲 T/2 的相位 = -w*T/2

printf('pi/ws = %.8f\n', pi/ws);
printf('T/2   = %.8f\n', T/2);
printf('兩者差異 = %.2e\n\n', abs(pi/ws - T/2));
printf('兩條相位曲線最大差異 = %.2e 度\n', max(abs(pha_zoh_linear - pha_pure_delay)));
printf('=> ZOH 等效引入 T/2 = %g 秒的純延遲\n', T/2);

figure('Position', [50 50 800 380]);
plot(w/ws, pha_zoh_linear, 'b-', 'LineWidth', 2); hold on;
plot(w/ws, pha_pure_delay, 'r--', 'LineWidth', 2); grid on;
title('ZOH 的線性相位 = 純延遲 T/2 的相位');
xlabel('\omega / \omega_s'); ylabel('相位 (度)');
legend('ZOH 線性相位項 -\pi\omega/\omega_s', '純延遲 T/2 的相位 -\omega T/2', ...
       'Location', 'southwest');

### 結果解讀：這對控制系統設計的意義

兩條線完全重合（差異 $10^{-13}$ 度等級）。

> **💡 一句話**：只要在迴路裡放一個取樣器加 ZOH，你就**免費得到了一個 $T/2$ 秒的延遲**——而延遲是控制系統的天敵，它會吃掉相位裕度、讓系統趨向不穩定。

**具體影響**：在頻率 $\omega$ 處，這個延遲造成的額外相位落後是 $\omega T/2$ 弧度。以本例 $T=0.1$ 秒為例：

| 頻率 | 額外相位落後 |
|---|---|
| $\omega=1$ rad/s | $2.9^\circ$ |
| $\omega=10$ rad/s | $28.6^\circ$ |
| $\omega=\omega_s/2=31.4$ rad/s | $90^\circ$ |

**這給了「$T$ 該取多小」第二個判準**：

1. **Shannon 定理的判準**（本章第 3.6 節）：$\omega_s$ 要大於訊號最高頻率的兩倍，否則資訊遺失。
2. **相位裕度的判準**（本節）：$T/2$ 的延遲在系統的**增益交越頻率**處造成的相位落後，必須小到系統仍有足夠的相位裕度。

實務上第 2 個判準通常**更嚴格**，這也是為什麼工程慣例是「每個閉迴路振盪週期取樣 10～20 次」，而不是 Shannon 定理的最低要求 2 次。

---


## 十、本章總結

### 公式速查表

| 式號 | 公式 | 用途 | 對應實驗 |
|---|---|---|---|
| (3-2) | $\bar{E}(s)=E^*(s)\cdot\dfrac{1-e^{-Ts}}{s}$ | 取樣–保持的分解（全章核心） | 1, 2 |
| (3-3) | $E^*(s)=\sum_{n=0}^{\infty}e(nT)e^{-nTs}$ | 星號轉換定義 | 3 |
| (3-6) | $e^*(t)=e(t)\delta_T(t)$ | 取樣即脈衝調變 | 2 |
| (3-8) | $G_{h0}(s)=\dfrac{1-e^{-Ts}}{s}$ | ZOH 轉移函數 | 6 |
| (3-10) | 留數法 | 產生 z 轉換表 | — |
| (3-11) | $E^*(s)=\dfrac1T\sum_nE(s+jn\omega_s)+\dfrac{e(0)}2$ | 頻譜複製 | — |
| (3-20) | $E^*(s+jm\omega_s)=E^*(s)$ | 性質 1：週期性 | 4 |
| (3-22) | 頻譜以 $\omega_s$ 為間隔複製 | 混疊分析 | 4, 5 |
| — | Shannon 取樣定理 | $\omega_s>2\omega_{\max}$ | 5 |
| (3-32) | $| G_{h0}|=T|\mathrm{sinc}(\pi\omega/\omega_s)|$ | ZOH 振幅 | 6 |
| (3-33) | $\angle G_{h0}=-\pi\omega/\omega_s+\phi$ | ZOH 相位（等效 $T/2$ 延遲） | 6, 9 |
| (3-36) | $G_{h1}(s)=\dfrac{1+Ts}{T}\left[\dfrac{1-e^{-Ts}}{s}\right]^2$ | FOH 轉移函數 | 7 |
| (3-37)(3-38) | FOH 振幅與相位 | ZOH/FOH 比較 | 7 |
| (3-39) | $G_{hk}(s)$ | 分數階保持器 | 8 |

### MATLAB / Octave 常見錯誤

| 錯誤 | 症狀 | 正確做法 |
|---|---|---|
| `sin(x)/x`（少了點） | 結果變成一個純量，或維度錯誤 | **`sin(x)./x`** |
| `(...)^2` 對向量 | `nonconformant arguments` 錯誤 | **`(...).^2`** |
| 頻率軸從 0 開始 | sinc 出現 `NaN`，圖上斷點 | 從 `1e-6` 開始，或事後補 `sc(w==0)=1` |
| 用 `plot` 畫 ZOH 輸出 | 看起來像連續變化，誤導 | **`stairs`** |
| 用 `plot` 畫脈衝串 | 把脈衝連成折線，意義全錯 | **`stem`** |
| 寫 `e^(-T*s)` | 語法錯誤（`e` 未定義） | **`exp(-T*s)`** |
| `angle()` 忘了轉度數 | 圖的 y 軸是 $\pm\pi$ 而非 $\pm180$ | `angle(G)*180/pi` |
| Octave 忘記 `pkg load control` | `'c2d' undefined` | 開頭載入（本章其實用不到，但下一章需要） |

### 承先啟後

**本章建立的最重要成果**，是 Fig. 3-5 那個「理想取樣器 + $\dfrac{1-e^{-Ts}}{s}$」的數學模型——**所有數位控制系統的分析與設計都以它為根本**。

同時本章也留下一個問題：**取樣器沒有轉移函數**，所以不能像連續系統那樣把方塊圖直接相乘化簡。

**第 4 章**就要解決這個問題：從星號轉換出發，推導開迴路離散系統的**脈衝轉移函數** $G(z)$，並建立 $E(z)=E^*(s)\big|_{e^{sT}=z}$ 這個關鍵橋樑——實驗 3 的結果已經預告了它。
